In [2]:
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
import torch
import numpy as np
import rasterio
from rasterio.mask import mask
import gc
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4
)

upsampler = RealESRGANer(
    scale=4,
    model_path="./data/realesgran/RealESRGAN_x4plus.pth",
    model=model,
    tile=128,         # 🔥 clave
    tile_pad=10,
    pre_pad=0,
    half=True if device.type == "cuda" else False,
    device=device
)

/app/.venv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def extract_tiles_overlap(img, tile_size=256, overlap=32):
    tiles = []
    H, W, C = img.shape
    
    step = tile_size - overlap
    
    for i in range(0, H, step):
        for j in range(0, W, step):
            tile = img[i:i+tile_size, j:j+tile_size]
            
            if tile.shape[0] < tile_size or tile.shape[1] < tile_size:
                pad_h = tile_size - tile.shape[0]
                pad_w = tile_size - tile.shape[1]
                tile = np.pad(tile, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
            
            tiles.append((tile, i, j))
    
    return tiles

In [4]:
def reconstruct_from_tiles(tiles, out_shape, scale=4, overlap=32):
    H, W, C = out_shape
    result = np.zeros((H*scale, W*scale, C), dtype=np.float32)
    weight = np.zeros((H*scale, W*scale, 1), dtype=np.float32)

    tile_size = tiles[0][0].shape[0]
    step = tile_size - overlap

    for tile, i, j, sr_tile in tiles:
        i_s = i * scale
        j_s = j * scale

        h, w = sr_tile.shape[:2]

        result[i_s:i_s+h, j_s:j_s+w] += sr_tile
        weight[i_s:i_s+h, j_s:j_s+w] += 1

    return result / np.maximum(weight, 1e-6)

In [ ]:
def superres_image(img, upsampler, tile_size=256, overlap=32):
    
    tiles = extract_tiles_overlap(img, tile_size, overlap)
    sr_tiles = []

    for tile, i, j in tiles:
        try:
            sr, _ = upsampler.enhance(tile, outscale=4)
            sr = sr.astype(np.float32) / 255.0
        except Exception as e:
            print("error en tile:", e)
            continue

        sr_tiles.append((tile, i, j, sr))

    result = reconstruct_from_tiles(
        sr_tiles,
        out_shape=img.shape,
        scale=4,
        overlap=overlap
    )

    return result

: 

In [ ]:
with rasterio.open("./data/composite/MGRS-18NUH/MGRS-18NUH_composite.tif") as src:
    composite = src.read()  # (bands, height, width)
    profile = src.profile
# preparar RGB
rgb = composite[[2,1,0]]
rgb = np.transpose(rgb, (1,2,0))

rgb = rgb / 10000.0
rgb = np.clip(rgb, 0, 1)

img = (rgb * 255).astype(np.uint8)

# 🔥 aplicar superres
rgb_sr = superres_image(img, upsampler)